In [1]:
from tqdm.auto import tqdm
tqdm.pandas()

import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if not module_path in sys.path:
    sys.path.insert(0, module_path)

from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.wrangling_tools import is_non_empty
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps
from innoprod.text_analysis.code_analysis import CodeAnalysis

In [2]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

In [3]:
cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis',
    'Key potential industry 4.0 solutions to address the identified problems/gaps',
    'Recommended Action Plan to utilise Industry 4.0 Technology',
    'Overview of qualitative benefits of recommended Action Plan (positive/negative)',
    'Skills and other requirements that will be needed to successfully implement the recommended Action Plan',
    'What has been your overall opinion of the support you have received in this programme? (Add comments)'
]

# All question responses as long table

In [4]:
responses_df = roadmaps_df[['Client ID'] + cols].melt(id_vars=['Client ID'], value_vars=cols, var_name='Question', value_name='Response')
responses_df = responses_df[is_non_empty(responses_df['Response'])]

responses_df

,Client ID,Question,Response
0,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,[REDACTED] business has completed an edge digi...
1,458079bc-a5ab-2055-d514-6733331e9c5f,Summary review of Edge Digital diagnostic repo...,[REDACTED] Score: 7 STATUS: Based on your resp...
2,0369B4F9-9E83-E83D-9E6B-BF82E264A2BA,Summary review of Edge Digital diagnostic repo...,The EDD has identified that the Company has in...
3,e9b5b5a2-1ba0-1d3a-a374-67ed061c1e40,Summary review of Edge Digital diagnostic repo...,Summary review of [REDACTED] diagnostic report...
4,052CB881-3557-DCFA-E472-0E55A5D04590,Summary review of Edge Digital diagnostic repo...,The business has rated itself at level 4 in te...
...,...,...,...
2414,3044E4BE-D41D-1AD3-7DFE-D22F7E559972,What has been your overall opinion of the supp...,No negatives to report. Very good process and ...
2415,0EC71AFB-5867-DE5F-5061-32AEAE4B24BF,What has been your overall opinion of the supp...,Good. The only thing to say is it is difficult...
2416,2346FC83-5B42-B90D-3BBB-16BFD156902E,What has been your overall opinion of the supp...,"as usual a simple process, managed for us by o..."
2417,2AA6320B-75FF-BABC-1E79-EB96B0AE650E,What has been your overall opinion of the supp...,excellent support as always. I had received su...


# Break into sentences

In [5]:
from nltk.tokenize import PunktSentenceTokenizer
tokenizer = PunktSentenceTokenizer()

sentences_df = responses_df.copy()
sentences_df['Sentences'] = sentences_df.apply(lambda row: tokenizer.tokenize(row['Response']), axis=1)
sentences_df = sentences_df[['Client ID', 'Question', 'Sentences']].explode('Sentences').reset_index().rename(columns={'index' : 'Sentence Number', 'Sentences' : 'Sentence'})
sentences_df['Sentence Number'] = sentences_df.groupby('Sentence Number').cumcount() + 1
sentences_df

,Sentence Number,Client ID,Question,Sentence
0,1,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,[REDACTED] business has completed an edge digi...
1,2,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,They are aware of [REDACTED] weaknesses in not...
2,3,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,[REDACTED] company has invested in appropriate...
3,4,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,They also recognise that they struggle to util...
4,5,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,[REDACTED] key improvement area at this time i...
...,...,...,...,...
8027,2,0EC71AFB-5867-DE5F-5061-32AEAE4B24BF,What has been your overall opinion of the supp...,The only thing to say is it is difficult somet...
8028,1,2346FC83-5B42-B90D-3BBB-16BFD156902E,What has been your overall opinion of the supp...,"as usual a simple process, managed for us by o..."
8029,1,2AA6320B-75FF-BABC-1E79-EB96B0AE650E,What has been your overall opinion of the supp...,excellent support as always.
8030,2,2AA6320B-75FF-BABC-1E79-EB96B0AE650E,What has been your overall opinion of the supp...,I had received support from Marcus and Oxford ...


# Pre-process sentences

In [6]:
sentences_df['Cleaned Sentence'] = sentences_df['Sentence'].str.lower()

# TODO Standardise acronyms? 
# Need to be very careful if removing stop words, as the standard stop word lists contains words like "not" that are important here.
# Consider lemmatisation/stemming?

# Count repeated sentences

In [7]:
recurring_sentences_df = sentences_df['Cleaned Sentence'].value_counts().to_frame().reset_index().rename(columns={'count': 'Count'})
# recurring_sentences_df = recurring_sentences_df.merge(sentences_df[['Cleaned Sentence', 'Question']], on='Cleaned Sentence', how='left').drop_duplicates(subset=['Cleaned Sentence', 'Question'])
# recurring_sentences_df = recurring_sentences_df.groupby(['Cleaned Sentence', 'Count'])['Question'].apply(list).to_frame().rename(columns={'Question': 'Questions'}).reset_index().sort_values(by='Count', ascending=False)
# recurring_sentences_df['Num Questions'] = recurring_sentences_df['Questions'].apply(len)
recurring_sentences_df

,Cleaned Sentence,Count
0,the development of a formal digital strategy s...,78
1,impact of the implementation of the action pla...,59
2,as with many business one of the main barriers...,55
3,this will require investment in specialist equ...,52
4,this will be underpinned by increased function...,50
...,...,...
4550,the only thing to say is it is difficult somet...,1
4551,"as usual a simple process, managed for us by o...",1
4552,excellent support as always.,1
4553,i had received support from marcus and oxford ...,1


# Manual iterative pattern matching

In [8]:
code_analysis = CodeAnalysis(recurring_sentences_df)

In [9]:
s = code_analysis.get_next_sentence()
s

'the development of a formal digital strategy should be the starting point for this and feature prioritised actions underpinned by formal project plans to support implementation.'

In [13]:
code_analysis.add_keyword('digital strategy', 'Digital Strategy')
code_analysis.add_keyword('project plans to support implementation', 'Implementation Project Plan(s)')
code_analysis.add_quantifier('development of a formal KEYWORD should be the starting point', 'Absence')

In [14]:
code_analysis.display_sentence_with_highlights(s)

HTML(value='the <span style="background-color: lightblue;">development of a formal <span style="background-col…

In [12]:
code_analysis.mark_sentence_analysed(s)